In [ ]:
import os
import shutil
from PIL import Image
from google.colab import drive

# Mount Google Drive to access files
drive.mount('/content/drive')

# Define paths and resize dimensions
zip_file_path = "/content/drive/My Drive/Pneumonia Complete Data.zip"
extract_path = "/content/dataset"
output_zip_path = "/content/drive/My Drive/Pneumonia_Processed.zip"
target_size = (256, 256)  # All images will be resized to 256x256

# Extract the ZIP file to the working directory
shutil.unpack_archive(zip_file_path, extract_path)

# Set folder names and paths for each category
dataset_path = os.path.join(extract_path, "Pneumonia Complete Dataset")
categories = ["Bacterial Pneumonia", "Viral Pneumonia", "NORMAL"]

# Loop through each category folder, rename and resize images
for category in categories:
    folder_path = os.path.join(dataset_path, category)
    output_category_path = os.path.join(extract_path, f"Processed_{category}")

    os.makedirs(output_category_path, exist_ok=True)

    images = sorted(os.listdir(folder_path))  # Sort to maintain consistent order
    for idx, img_name in enumerate(images):
        img_path = os.path.join(folder_path, img_name)
        new_img_name = f"{category} ({idx+1}).jpg"
        new_img_path = os.path.join(output_category_path, new_img_name)

        try:
            with Image.open(img_path) as img:
                img = img.resize(target_size)
                img.save(new_img_path, "JPEG")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

# Create a new ZIP archive with the processed images
shutil.make_archive(output_zip_path.replace('.zip', ''), 'zip', extract_path)

print("Process Completed! New ZIP saved in Google Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Process Completed! New ZIP saved in Google Drive.


In [ ]:
import os
import shutil
import random
import zipfile
import cv2
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define Paths
zip_file_path = "/content/drive/My Drive/Pneumonia_Processed.zip"
extract_folder = "/content/Pneumonia_Processed"

# Extract the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)
print(" ZIP Extraction Completed!")

# Set Target Folders
all_folders = os.listdir(extract_folder)
processed_folders = [f for f in all_folders if f.startswith("Processed_")]

# Define Base Paths
output_base_path = "/content/Pneumonia_80_20_Splitted_Updated"
train_path = os.path.join(output_base_path, "train")
test_path = os.path.join(output_base_path, "test")
os.makedirs(train_path, exist_ok=True)
os.makedirs(test_path, exist_ok=True)

# Define Train-Test Ratio and Resize Size
train_ratio = 0.8
target_size = (224, 224)

# Split and Resize
def split_and_resize_images():
    for folder_name in processed_folders:
        category_path = os.path.join(extract_folder, folder_name)
        if not os.path.isdir(category_path):
            continue

        images = os.listdir(category_path)
        random.shuffle(images)
        total = len(images)

        train_count = int(total * train_ratio)
        test_count = total - train_count

        train_images = images[:train_count]
        test_images = images[train_count:]

        # Create category folders
        train_cat_path = os.path.join(train_path, folder_name)
        test_cat_path = os.path.join(test_path, folder_name)
        os.makedirs(train_cat_path, exist_ok=True)
        os.makedirs(test_cat_path, exist_ok=True)

        # Process Train Images
        for img_name in tqdm(train_images, desc=f"Train - {folder_name}"):
            img_path = os.path.join(category_path, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, target_size)
                cv2.imwrite(os.path.join(train_cat_path, img_name), img)

        # Process Test Images
        for img_name in tqdm(test_images, desc=f"Test - {folder_name}"):
            img_path = os.path.join(category_path, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.resize(img, target_size)
                cv2.imwrite(os.path.join(test_cat_path, img_name), img)

        print(f" {folder_name}: Train={train_count}, Test={test_count}")

# Run the Function
split_and_resize_images()

# Step 6: ZIP the Final Dataset
output_zip_path = "/content/Pneumonia_80_20_Splitted_Updated.zip"
with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(output_base_path):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, output_base_path))

# Step 7: Save Final ZIP to Google Drive
drive_save_path = "/content/drive/My Drive/Pneumonia_80_20_Splitted_Updated.zip"
shutil.move(output_zip_path, drive_save_path)

print(" Final Splitted ZIP saved in Google Drive!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 ZIP Extraction Completed!


Test - Processed_Viral Pneumonia: 100%|██████████| 1012/1012 [00:01<00:00, 971.96it/s]


 Processed_Viral Pneumonia: Train=4044, Test=1012


Test - Processed_NORMAL: 100%|██████████| 1078/1078 [00:01<00:00, 952.18it/s]


 Processed_NORMAL: Train=4309, Test=1078


Test - Processed_Bacterial Pneumonia: 100%|██████████| 1449/1449 [00:01<00:00, 952.40it/s]


 Processed_Bacterial Pneumonia: Train=5794, Test=1449
 Final Splitted ZIP saved in Google Drive!


In [ ]:
import os
import shutil
from PIL import Image
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Paths
zip_file_path = "/content/drive/My Drive/Pneumonia Balanced Dataset.zip"
extract_path = "/content/dataset_balanced"
output_zip_path = "/content/drive/My Drive/Pneumonia_Processed_Balanced.zip"
target_size = (256, 256)

# Unzip the dataset
shutil.unpack_archive(zip_file_path, extract_path)

# Folder names
dataset_path = os.path.join(extract_path, "Pneumonia Complete Dataset")
categories = ["Bacterial Pneumonia", "Viral Pneumonia", "NORMAL"]

# Resize and rename images
for category in categories:
    input_folder = os.path.join(dataset_path, category)
    output_folder = os.path.join(extract_path, f"Processed_{category}")
    os.makedirs(output_folder, exist_ok=True)

    images = sorted(os.listdir(input_folder))
    for index, img_name in enumerate(images):
        img_path = os.path.join(input_folder, img_name)
        new_name = f"{category} ({index + 1}).jpg"
        new_path = os.path.join(output_folder, new_name)

        try:
            with Image.open(img_path) as img:
                img = img.resize(target_size)
                img.save(new_path, "JPEG")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

# Create new zip
shutil.make_archive(output_zip_path.replace(".zip", ""), 'zip', extract_path)

print("All images processed and ZIP saved successfully in Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All images processed and ZIP saved successfully in Drive.


In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
import tensorflow as tf
print("GPU Available: ", tf.config.list_physical_devices('GPU'))


GPU Available:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.0 MB/s eta 0:00:00


In [ ]:
pip install git+https://github.com/openai/CLIP.git


  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-qcsxtiq_
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-qcsxtiq_
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
#  Deep Learning & Image Processing
import tensorflow as tf
import keras
import cv2
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#  File & Data Handling
import os
import zipfile
import shutil

#  For Data Loading & Augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

#  Model Evaluation
from sklearn.metrics import classification_report, confusion_matrix

#  PDF Report Generation
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader

#  CLIP Model (We’ll load later)
import torch
import clip
from PIL import Image

print("All libraries loaded successfully!")


All libraries loaded successfully!


In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
zip_file_path = "/content/drive/My Drive/Pneumonia_ReBalanced_Dataset.zip"
extract_path = "/content/dataset"

# Extract the ZIP file
shutil.unpack_archive(zip_file_path, extract_path)
print(" Dataset successfully extracted!")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Dataset successfully extracted!


In [ ]:
import os

# Extracted dataset ka path jahan aapne unzip kiya tha
dataset_path = "/content/dataset"

# Check karna ke is path mein kya kya maujood hai
print(" Extracted dataset contains:")
for item in os.listdir(dataset_path):
    item_path = os.path.join(dataset_path, item)
    if os.path.isdir(item_path):
        print(f" {item}/")
        # Har folder ke andar bhi check kar lein
        for subitem in os.listdir(item_path)[:5]:  # sirf pehle 5 items dikhayen
            print(f"   └── {subitem}")
    else:
        print(f" {item}")


 Extracted dataset contains:
 Processed_Bacterial Pneumonia/
   └── Bacterial Pneumonia (3153).jpg
   └── Bacterial Pneumonia (2311).jpg
   └── Bacterial Pneumonia (459).jpg
   └── Bacterial Pneumonia (3532).jpg
   └── Bacterial Pneumonia (2025).jpg
 Processed_NORMAL/
   └── NORMAL (3324).jpg
   └── NORMAL (2563).jpg
   └── NORMAL (1220).jpg
   └── NORMAL (28).jpg
   └── NORMAL (3688).jpg
 Processed_Viral Pneumonia/
   └── Viral Pneumonia (1941).jpg
   └── Viral Pneumonia (1738).jpg
   └── Viral Pneumonia (978).jpg
   └── Viral Pneumonia (2951).jpg
   └── Viral Pneumonia (4428).jpg


In [ ]:
import os

base_path = '/content/dataset'
folders = os.listdir(base_path)
print(" Folders Inside Dataset :")
for folder in folders:
    print("-", folder)


 Folders Inside Dataset :
- Processed_Bacterial Pneumonia
- Processed_NORMAL
- Processed_Viral Pneumonia


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Poori dataset ko training ke liye lena hai
datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = datagen.flow_from_directory(
    '/content/dataset',           # Tumhara original extracted folder
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)


Found 15000 images belonging to 3 classes.


In [ ]:
import tensorflow as tf
tf.config.experimental.list_physical_devices('GPU')


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
# -------------------------------
#  CNN Model + Smart Inference
# -------------------------------
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

# Define CNN Architecture
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes: Bacterial, Normal, Viral
])

# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Summary
model.summary()

# Optional EarlyStopping
early_stop = EarlyStopping(
    monitor='accuracy',
    patience=3,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    train_generator,
    epochs=20,
    steps_per_epoch=len(train_generator),
    callbacks=[early_stop]
)

print(" Model training completed!")

# ------------------------------------------
#  Inference Function with Smart Validation
# ------------------------------------------
# Labels for output
class_labels = ['Bacterial Pneumonia', 'Normal', 'Viral Pneumonia']

def predict_image_with_validation(model, image_path, threshold=0.6):
    try:
        img = load_img(image_path, target_size=(224, 224))
        img_array = img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        # Predict
        preds = model.predict(img_array)
        confidence = np.max(preds)
        predicted_class = np.argmax(preds)

        print(" Prediction Probabilities:", preds)
        print(" Highest Confidence:", confidence)

        if confidence < threshold:
            print("  Suspicious Image: This might not be a valid chest X-ray.")
        else:
            label = class_labels[predicted_class]
            print(f" Prediction: {label} (Confidence: {confidence:.2f})")

    except Exception as e:
        print(" Error processing image:", e)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,245,443 (84.86 MB)

 Trainable params: 22,245,443 (84.86 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


FailedPreconditionError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.11/dist-pa

In [ ]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))


In [ ]:
model.add(Dense(128, activation='relu'))  # Instead of 256


In [ ]:
import tensorflow as tf
import keras
import cv2
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import clip
from PIL import Image
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader

print(" All libraries reloaded successfully!")


In [ ]:
import os
os.kill(os.getpid(), 9)
